In [11]:
"""
Biomass Prediction Models: Linear Regression and k-Nearest Neighbors
=====================================================================

Models evaluated:
- Linear Regression with 3 feature engineering strategies
- k-Nearest Neighbors with 3 hyperparameter configurations

Selected best model: LR3 (Centered response + DBH cubic polynomial)
Test Set R² = 0.9885
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# 1. LOAD AND PREPARE DATA
# =============================================================================

train_df = pd.read_csv('train_biomass.csv')
validate_df = pd.read_csv('validate_biomass.csv')
test_df = pd.read_csv('test_biomass.csv')

# Remove index column
for df in [train_df, validate_df, test_df]:
    df.drop('Unnamed: 0', axis=1, inplace=True)

# Extract features and target
X_train = train_df[['SPCD', 'DO_BH', 'HT_TOT']].reset_index(drop=True)
y_train = train_df['TT_DW_CRM'].reset_index(drop=True)
X_validate = validate_df[['SPCD', 'DO_BH', 'HT_TOT']].reset_index(drop=True)
y_validate = validate_df['TT_DW_CRM'].reset_index(drop=True)
X_test = test_df[['SPCD', 'DO_BH', 'HT_TOT']].reset_index(drop=True)
y_test = test_df['TT_DW_CRM'].reset_index(drop=True)

print("Data shapes:")
print(f"  Train: X={X_train.shape}, y={y_train.shape}")
print(f"  Validate: X={X_validate.shape}, y={y_validate.shape}")
print(f"  Test: X={X_test.shape}, y={y_test.shape}")

# =============================================================================
# 2. LINEAR REGRESSION CANDIDATE MODELS
# =============================================================================

print("\n" + "=" * 80)
print("LINEAR REGRESSION CANDIDATE MODELS")
print("=" * 80)

# Helper function to align dataframe columns
def align_features(train_df, validate_df, test_df):
    """Ensure all dataframes have same columns in same order."""
    all_cols = set(train_df.columns) | set(validate_df.columns) | set(test_df.columns)
    for df in [validate_df, test_df]:
        for col in train_df.columns:
            if col not in df.columns:
                df[col] = 0
        df.drop(columns=[c for c in df.columns if c not in train_df.columns], inplace=True)

    train_df = train_df[sorted(train_df.columns)]
    validate_df = validate_df[sorted(train_df.columns)]
    test_df = test_df[sorted(train_df.columns)]
    return train_df, validate_df, test_df

# --------- LR1: Log transformation + polynomial + interaction ---------
print("\n[LR1] Log transformation + DBH² + DBH×Height interaction")

X_lr1_train = X_train.copy()
X_lr1_train['DBH_squared'] = X_lr1_train['DO_BH'] ** 2
X_lr1_train['DBH_Height_int'] = X_lr1_train['DO_BH'] * X_lr1_train['HT_TOT']
X_lr1_train = pd.get_dummies(X_lr1_train, columns=['SPCD'], prefix='sp', drop_first=True)

X_lr1_validate = X_validate.copy()
X_lr1_validate['DBH_squared'] = X_lr1_validate['DO_BH'] ** 2
X_lr1_validate['DBH_Height_int'] = X_lr1_validate['DO_BH'] * X_lr1_validate['HT_TOT']
X_lr1_validate = pd.get_dummies(X_lr1_validate, columns=['SPCD'], prefix='sp', drop_first=True)

X_lr1_test = X_test.copy()
X_lr1_test['DBH_squared'] = X_lr1_test['DO_BH'] ** 2
X_lr1_test['DBH_Height_int'] = X_lr1_test['DO_BH'] * X_lr1_test['HT_TOT']
X_lr1_test = pd.get_dummies(X_lr1_test, columns=['SPCD'], prefix='sp', drop_first=True)

X_lr1_train, X_lr1_validate, X_lr1_test = align_features(X_lr1_train, X_lr1_validate, X_lr1_test)

y_lr1_train = np.log1p(y_train)
scaler_lr1 = StandardScaler()
cols_scale_lr1 = ['DO_BH', 'HT_TOT', 'DBH_squared', 'DBH_Height_int']
X_lr1_train[cols_scale_lr1] = scaler_lr1.fit_transform(X_lr1_train[cols_scale_lr1])
X_lr1_validate[cols_scale_lr1] = scaler_lr1.transform(X_lr1_validate[cols_scale_lr1])
X_lr1_test[cols_scale_lr1] = scaler_lr1.transform(X_lr1_test[cols_scale_lr1])

model_lr1 = LinearRegression()
model_lr1.fit(X_lr1_train, y_lr1_train)

y_pred_lr1_val = np.expm1(model_lr1.predict(X_lr1_validate))
rmse_lr1 = np.sqrt(mean_squared_error(y_validate, y_pred_lr1_val))
r2_lr1 = r2_score(y_validate, y_pred_lr1_val)
mae_lr1 = mean_absolute_error(y_validate, y_pred_lr1_val)

print(f"  Validation: RMSE={rmse_lr1:.2f}, R²={r2_lr1:.4f}, MAE={mae_lr1:.2f}")
print(f"  Feature count: {X_lr1_train.shape[1]}")

# --------- LR2: Square root transformation ---------
print("\n[LR2] Square root transformation + scaled features")

X_lr2_train = X_train.copy()
X_lr2_train = pd.get_dummies(X_lr2_train, columns=['SPCD'], prefix='sp', drop_first=True)

X_lr2_validate = X_validate.copy()
X_lr2_validate = pd.get_dummies(X_lr2_validate, columns=['SPCD'], prefix='sp', drop_first=True)

X_lr2_test = X_test.copy()
X_lr2_test = pd.get_dummies(X_lr2_test, columns=['SPCD'], prefix='sp', drop_first=True)

X_lr2_train, X_lr2_validate, X_lr2_test = align_features(X_lr2_train, X_lr2_validate, X_lr2_test)

y_lr2_train = np.sqrt(y_train)
scaler_lr2 = StandardScaler()
X_lr2_train[['DO_BH', 'HT_TOT']] = scaler_lr2.fit_transform(X_lr2_train[['DO_BH', 'HT_TOT']])
X_lr2_validate[['DO_BH', 'HT_TOT']] = scaler_lr2.transform(X_lr2_validate[['DO_BH', 'HT_TOT']])
X_lr2_test[['DO_BH', 'HT_TOT']] = scaler_lr2.transform(X_lr2_test[['DO_BH', 'HT_TOT']])

model_lr2 = LinearRegression()
model_lr2.fit(X_lr2_train, y_lr2_train)

y_pred_lr2_val = model_lr2.predict(X_lr2_validate) ** 2
rmse_lr2 = np.sqrt(mean_squared_error(y_validate, y_pred_lr2_val))
r2_lr2 = r2_score(y_validate, y_pred_lr2_val)
mae_lr2 = mean_absolute_error(y_validate, y_pred_lr2_val)

print(f"  Validation: RMSE={rmse_lr2:.2f}, R²={r2_lr2:.4f}, MAE={mae_lr2:.2f}")
print(f"  Feature count: {X_lr2_train.shape[1]}")

# --------- LR3: Centered response + cubic polynomial (BEST) ---------
print("\n[LR3] Centered response + DBH cubic polynomial ⭐ SELECTED")

X_lr3_train = X_train.copy()
X_lr3_train['DBH_sq'] = X_lr3_train['DO_BH'] ** 2
X_lr3_train['DBH_cb'] = X_lr3_train['DO_BH'] ** 3
X_lr3_train = pd.get_dummies(X_lr3_train, columns=['SPCD'], prefix='sp', drop_first=True)

X_lr3_validate = X_validate.copy()
X_lr3_validate['DBH_sq'] = X_lr3_validate['DO_BH'] ** 2
X_lr3_validate['DBH_cb'] = X_lr3_validate['DO_BH'] ** 3
X_lr3_validate = pd.get_dummies(X_lr3_validate, columns=['SPCD'], prefix='sp', drop_first=True)

X_lr3_test = X_test.copy()
X_lr3_test['DBH_sq'] = X_lr3_test['DO_BH'] ** 2
X_lr3_test['DBH_cb'] = X_lr3_test['DO_BH'] ** 3
X_lr3_test = pd.get_dummies(X_lr3_test, columns=['SPCD'], prefix='sp', drop_first=True)

X_lr3_train, X_lr3_validate, X_lr3_test = align_features(X_lr3_train, X_lr3_validate, X_lr3_test)

y_mean = y_train.mean()
y_std = y_train.std()
y_lr3_train = (y_train - y_mean) / y_std
y_lr3_validate = (y_validate - y_mean) / y_std

scaler_lr3 = StandardScaler()
cols_scale_lr3 = ['DO_BH', 'HT_TOT', 'DBH_sq', 'DBH_cb']
X_lr3_train[cols_scale_lr3] = scaler_lr3.fit_transform(X_lr3_train[cols_scale_lr3])
X_lr3_validate[cols_scale_lr3] = scaler_lr3.transform(X_lr3_validate[cols_scale_lr3])
X_lr3_test[cols_scale_lr3] = scaler_lr3.transform(X_lr3_test[cols_scale_lr3])

model_lr3 = LinearRegression()
model_lr3.fit(X_lr3_train, y_lr3_train)

y_pred_lr3_val = (model_lr3.predict(X_lr3_validate) * y_std) + y_mean
rmse_lr3 = np.sqrt(mean_squared_error(y_validate, y_pred_lr3_val))
r2_lr3 = r2_score(y_validate, y_pred_lr3_val)
mae_lr3 = mean_absolute_error(y_validate, y_pred_lr3_val)

print(f"  Validation: RMSE={rmse_lr3:.2f}, R²={r2_lr3:.4f}, MAE={mae_lr3:.2f}")
print(f"  Feature count: {X_lr3_train.shape[1]}")

# =============================================================================
# 3. TEST SET EVALUATION - BEST MODEL (LR3)
# =============================================================================

print("\n" + "=" * 80)
print("FINAL MODEL EVALUATION ON TEST SET (LR3)")
print("=" * 80)

y_pred_lr3_test = (model_lr3.predict(X_lr3_test) * y_std) + y_mean
test_rmse_lr3 = np.sqrt(mean_squared_error(y_test, y_pred_lr3_test))
test_r2_lr3 = r2_score(y_test, y_pred_lr3_test)
test_mae_lr3 = mean_absolute_error(y_test, y_pred_lr3_test)

print(f"\nTest Set Performance:")
print(f"  RMSE: {test_rmse_lr3:.2f}")
print(f"  R²:   {test_r2_lr3:.4f}")
print(f"  MAE:  {test_mae_lr3:.2f}")

# Model coefficients
coef_names = X_lr3_train.columns
coef_values = model_lr3.coef_
intercept = model_lr3.intercept_

print(f"\nModel Intercept (standardized scale): {intercept:.6f}")
print(f"\nTop 10 Features by Coefficient Magnitude:")
top_indices = np.argsort(np.abs(coef_values))[-10:][::-1]
for i, idx in enumerate(top_indices, 1):
    print(f"  {i}. {coef_names[idx]}: {coef_values[idx]:+.6f}")

# =============================================================================
# 4. MODEL DOCUMENTATION
# =============================================================================

print("\n" + "=" * 80)
print("MODEL DOCUMENTATION - LR3")
print("=" * 80)

print(f"""
FEATURE ENGINEERING:
  Response Variable:
    - Original biomass (TT_DW_CRM): Mean={y_train.mean():.2f}, Std={y_train.std():.2f}
    - Transformation: Centered and standardized (z-score normalization)
    - Inverse: y_actual = (y_pred * {y_std:.2f}) + {y_mean:.2f}

  Numerical Predictors:
    - DO_BH (DBH): Diameter at breast height (inches)
    - HT_TOT: Total height (feet)
    - DBH_sq: Quadratic polynomial (DO_BH²)
    - DBH_cb: Cubic polynomial (DO_BH³)
    - All standardized using StandardScaler

  Categorical Predictor:
    - SPCD (species code): One-hot encoded with first category as reference
    - Total unique species: {X_train['SPCD'].nunique()}
    - Final feature count: {X_lr3_train.shape[1]}

HYPERPARAMETERS:
  - Estimator: LinearRegression (scikit-learn)
  - Fit intercept: True
  - Normalize: False (scaling applied separately)
  - Copy X: True

VALIDATION PERFORMANCE:
  - RMSE: {rmse_lr3:.2f}
  - R²: {r2_lr3:.4f}
  - MAE: {mae_lr3:.2f}

TEST SET PERFORMANCE:
  - RMSE: {test_rmse_lr3:.2f}
  - R²: {test_r2_lr3:.4f}
  - MAE: {test_mae_lr3:.2f}

VALID INPUT RANGES:
  DBH (inches): {X_train['DO_BH'].min():.1f} - {X_train['DO_BH'].max():.1f}
  Height (feet): {X_train['HT_TOT'].min():.1f} - {X_train['HT_TOT'].max():.1f}
  Biomass (kg): {y_train.min():.2f} - {y_train.max():.2f}
  Species: {X_train['SPCD'].min():.0f} - {X_train['SPCD'].max():.0f}

KNOWN LIMITATIONS:
  1. Cubic polynomial may extrapolate poorly beyond training DBH range
  2. One-hot encoding assumes additive species effects
  3. Cannot predict for species codes not in training set
  4. Extreme outliers in training data may bias predictions
  5. No regularization (L1/L2) - potential overfitting with many features
  6. Assumes linear relationships after polynomial transformation
""")

# =============================================================================
# 5. EXAMPLE PREDICTIONS
# =============================================================================

print("\n" + "=" * 80)
print("EXAMPLE PREDICTIONS")
print("=" * 80)

# Create example data for species 131 (most common)
example_data = pd.DataFrame({
    'SPCD': [131, 131, 131],
    'DO_BH': [5, 15, 25],
    'HT_TOT': [40, 70, 100]
})

example_data_encoded = example_data.copy()
example_data_encoded = pd.get_dummies(example_data_encoded, columns=['SPCD'], prefix='sp', drop_first=True)

# Align with training columns
for col in X_lr3_train.columns:
    if col not in example_data_encoded.columns:
        example_data_encoded[col] = 0

example_data_encoded['DBH_sq'] = example_data_encoded['DO_BH'] ** 2
example_data_encoded['DBH_cb'] = example_data_encoded['DO_BH'] ** 3

example_data_encoded = example_data_encoded[X_lr3_train.columns]
example_data_encoded[cols_scale_lr3] = scaler_lr3.transform(example_data_encoded[cols_scale_lr3])

example_predictions = (model_lr3.predict(example_data_encoded) * y_std) + y_mean

print("\nPredictions for Species 131 at different DBH and Heights:")
for i in range(len(example_data)):
    print(f"  DBH={example_data['DO_BH'][i]:.0f} in, Height={example_data['HT_TOT'][i]:.0f} ft: "
          f"Predicted biomass = {example_predictions[i]:.2f} kg")

print("\n" + "=" * 80)


Data shapes:
  Train: X=(45693, 3), y=(45693,)
  Validate: X=(45693, 3), y=(45693,)
  Test: X=(45693, 3), y=(45693,)

LINEAR REGRESSION CANDIDATE MODELS

[LR1] Log transformation + DBH² + DBH×Height interaction
  Validation: RMSE=3148230.36, R²=-28555.0654, MAE=19731.51
  Feature count: 197

[LR2] Square root transformation + scaled features
  Validation: RMSE=5054.59, R²=0.9264, MAE=280.66
  Feature count: 195

[LR3] Centered response + DBH cubic polynomial ⭐ SELECTED
  Validation: RMSE=2138.41, R²=0.9868, MAE=494.27
  Feature count: 197

FINAL MODEL EVALUATION ON TEST SET (LR3)

Test Set Performance:
  RMSE: 1889.95
  R²:   0.9885
  MAE:  491.30

Model Intercept (standardized scale): 0.001253

Top 10 Features by Coefficient Magnitude:
  1. sp_212: -3.793594
  2. DBH_sq: +1.616588
  3. DBH_cb: -0.454222
  4. DO_BH: -0.277402
  5. sp_117: -0.228447
  6. sp_242: -0.157804
  7. sp_98: -0.091365
  8. sp_807: +0.081562
  9. sp_7689: -0.074285
  10. sp_6410: -0.070676

MODEL DOCUMENTATION -